# Inserting, updating and deleting safely

Practice notebook for the datafromzero SQL lesson **Inserting, updating and deleting safely**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

You change only the small `stock` tables you create here. The cafe-sales tables are never changed, and the setup cell gives you a clean start.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it. Run it again at any time to start over.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## The stock table (done for you)

One row per product and store, with a key and a `CHECK`, as in the lesson *Creating tables and constraints*. Store 1 starts with Espresso (product 1), Croissant (10) and Cold Brew (6).

In [ ]:
con.sql("""
CREATE TABLE stock (
  store_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  units_on_hand INTEGER NOT NULL CHECK (units_on_hand >= 0),
  PRIMARY KEY (store_id, product_id)
);
INSERT INTO stock VALUES (1, 1, 40), (1, 10, 25), (1, 6, 12);
SELECT * FROM stock ORDER BY product_id
""").show()

## Worked example

### Step 1. Insert one row, then many

In [ ]:
con.sql("""
INSERT INTO stock VALUES (1, 7, 18);
INSERT INTO stock SELECT 2, product_id, 10 FROM stock WHERE store_id = 1;
SELECT * FROM stock ORDER BY store_id, product_id
""").show()

### Step 2. Preview, then update

In [ ]:
con.sql("""
SELECT * FROM stock WHERE store_id = 1 AND product_id = 10
""").show()

In [ ]:
con.sql("""
UPDATE stock SET units_on_hand = units_on_hand + 24 WHERE store_id = 1 AND product_id = 10
""").show()

In [ ]:
con.sql("""
SELECT * FROM stock WHERE store_id = 1 AND product_id = 10
""").show()

### Step 3. Delete the same way

In [ ]:
con.sql("""
SELECT * FROM stock WHERE store_id = 1 AND product_id = 7
""").show()

In [ ]:
con.sql("""
DELETE FROM stock WHERE store_id = 1 AND product_id = 7
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS rows_left FROM stock
""").show()

### Step 4. Make a mistake, then undo it

In [ ]:
con.sql("""
BEGIN
""").show()

In [ ]:
con.sql("""
DELETE FROM stock
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS rows_left FROM stock
""").show()

In [ ]:
con.sql("""
ROLLBACK
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS rows_left FROM stock
""").show()

### Step 5. Two tills, one count

`con.cursor()` gives a second session on the same database. Ana and Ben each get one. Run the cells in order. The one marked with an error is meant to fail: it prints the error and Run all keeps going.

In [ ]:
ana = con.cursor()
ben = con.cursor()

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ana.sql("""
    BEGIN;
    UPDATE stock SET units_on_hand = units_on_hand - 1 WHERE store_id = 1 AND product_id = 10;
    SELECT units_on_hand FROM stock WHERE store_id = 1 AND product_id = 10
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ben.sql("""
    SELECT units_on_hand FROM stock WHERE store_id = 1 AND product_id = 10
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    # This cell prints an error: Ben changes the row Ana has not committed yet
    ben.sql("""
    BEGIN;
    UPDATE stock SET units_on_hand = units_on_hand - 1 WHERE store_id = 1 AND product_id = 10
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ben.sql("ROLLBACK").show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ana.sql("COMMIT").show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ben.sql("""
    BEGIN;
    UPDATE stock SET units_on_hand = units_on_hand - 1 WHERE store_id = 1 AND product_id = 10;
    COMMIT
    """).show()
except Exception as e:
    print(e)


In [ ]:
con.sql("""
SELECT units_on_hand FROM stock WHERE store_id = 1 AND product_id = 10
""").show()

### Step 6. Reload one store, safely

`campus_count` is a staging table with Campus's new counts. Run the reload cell twice.

In [ ]:
con.sql("""
CREATE TABLE campus_count (store_id INTEGER, product_id INTEGER, units_on_hand INTEGER);
INSERT INTO campus_count VALUES (2, 1, 30), (2, 10, 14), (2, 6, 9);
SELECT * FROM campus_count
""").show()

In [ ]:
con.sql("""
BEGIN;
DELETE FROM stock WHERE store_id = 2;
INSERT INTO stock SELECT * FROM campus_count;
COMMIT
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS products, SUM(units_on_hand) AS units FROM stock WHERE store_id = 2
""").show()

### Step 7. Verify

In [ ]:
con.sql("""
SELECT store_id, COUNT(*) AS products, SUM(units_on_hand) AS units
FROM stock GROUP BY store_id ORDER BY store_id
""").show()

## Exercise 1. Campus sold 4 Espressos

Done for you: `stock_ex1` holds Espresso (product 1) at both store 1 and store 2, and the preview of the row to change. Write the `UPDATE` that subtracts 4 for that one row, then select the row to check it.

In [ ]:
con.sql("""
CREATE TABLE stock_ex1 (
  store_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  units_on_hand INTEGER NOT NULL CHECK (units_on_hand >= 0),
  PRIMARY KEY (store_id, product_id)
);
INSERT INTO stock_ex1 VALUES (1, 1, 40), (2, 1, 10), (2, 6, 10), (2, 10, 10);
SELECT * FROM stock_ex1 WHERE store_id = 2 AND product_id = 1
""").show()

In [ ]:
# The UPDATE: subtract 4 from Campus Espresso

In [ ]:
# Check the row

## Exercise 2. Reload Campus's counts from a staging table

Done for you: `stock_ex2` and `campus_count_ex2`. The staging table has four rows, typed from a paper sheet. Look at it with a `SELECT` first. Before you run the reload, write one sentence in the next cell: what will happen when you run the reload from step 6, and what will Campus's rows look like afterward? If the reload fails, run `ROLLBACK` before anything else.

In [ ]:
con.sql("""
CREATE TABLE stock_ex2 (
  store_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  units_on_hand INTEGER NOT NULL CHECK (units_on_hand >= 0),
  PRIMARY KEY (store_id, product_id)
);
INSERT INTO stock_ex2 VALUES (1, 1, 40), (2, 1, 10), (2, 6, 10), (2, 10, 10);
CREATE TABLE campus_count_ex2 (store_id INTEGER, product_id INTEGER, units_on_hand INTEGER);
INSERT INTO campus_count_ex2 VALUES (2, 1, 30), (2, 10, 14), (2, 6, 9), (2, 10, 14);
SELECT * FROM campus_count_ex2
""").show()

In [ ]:
# My expected result, in words:
#

In [ ]:
# The reload (BEGIN, DELETE, INSERT, COMMIT) against stock_ex2 and campus_count_ex2

In [ ]:
# ROLLBACK, if the reload failed

In [ ]:
# Check: COUNT(*) AS products, SUM(units_on_hand) AS units for store 2

In [ ]:
# The fixed reload, run twice, and the check again

## Exercise 3. Spot the flaw

Done for you: `stock_ex3` has 49 croissants at store 1. A colleague's till program reads the count, subtracts 1 in the program and writes the new number back. Ana's till and Ben's till both sell one at the same moment. Run the cells, then find out what went wrong and write the fix.

In [ ]:
con.sql("""
CREATE TABLE stock_ex3 (
  store_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  units_on_hand INTEGER NOT NULL CHECK (units_on_hand >= 0),
  PRIMARY KEY (store_id, product_id)
);
INSERT INTO stock_ex3 VALUES (1, 10, 49);
SELECT * FROM stock_ex3
""").show()

In [ ]:
ana3 = con.cursor()
ben3 = con.cursor()

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ana3.sql("SELECT units_on_hand FROM stock_ex3 WHERE store_id = 1 AND product_id = 10").show()
    ben3.sql("SELECT units_on_hand FROM stock_ex3 WHERE store_id = 1 AND product_id = 10").show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    ana3.sql("UPDATE stock_ex3 SET units_on_hand = 48 WHERE store_id = 1 AND product_id = 10").show()
    ben3.sql("UPDATE stock_ex3 SET units_on_hand = 48 WHERE store_id = 1 AND product_id = 10").show()
except Exception as e:
    print(e)


In [ ]:
con.sql("""
SELECT units_on_hand FROM stock_ex3 WHERE store_id = 1 AND product_id = 10
""").show()

In [ ]:
# What went wrong? Reset the count to 49, then write both sales the safe way